# Thai wooden frog — learned monocular depth (Colab)

This runs the one pipeline stage that will not run on the project laptop:
**monocular depth estimation with Depth Anything V2**. Segmentation, meshing and
rendering all already run locally; only this needs `torch`.

**What you get back:** `model3d/frog.obj` + `.mtl` + texture, ready to feed
straight into `render3d.py` at home.

**What this is not.** A single photograph recovers only the surface facing the
camera, so the result is a **relief, not a closed surface**. There is nothing
behind it. A full 360° turntable render is only honest after the multi-view
capture in `CAPTURE.md`.

## 1 · Runtime

A GPU makes this quicker but is not required — Depth Anything V2 Small runs on
CPU in about a minute. If you want one: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi -L 2>/dev/null || echo "No GPU attached — CPU is fine, just slower."

## 2 · Get the code

⚠️ **Push your local commits first.** This clones from GitHub, so any fix that
only exists on your laptop will not be here. The next cell checks for the two
that matter and stops if they are missing.

In [ ]:
import pathlib, subprocess

REPO = "https://github.com/ZweNyanWin/computer-vision-term-project.git"
if not pathlib.Path("computer-vision-term-project").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
%cd computer-vision-term-project

In [ ]:
import pathlib

source = pathlib.Path("reconstruct.py").read_text(encoding="utf-8")
required = {
    "orient_depth": "depth-orientation fix (stops the relief coming out hollow)",
    "_otsu_separability": "saturation segmentation (handles an unevenly lit backdrop)",
}
missing = {name: why for name, why in required.items() if f"def {name}" not in source}
if missing:
    raise SystemExit(
        "This clone is out of date. Missing:\n"
        + "\n".join(f"  - {name}: {why}" for name, why in missing.items())
        + "\n\nCommit and push on your laptop, then re-run the clone cell."
    )
print("Clone has both fixes. Good to continue.")

## 3 · Install

`requirements-depth.txt` pulls in torch, transformers and Pillow. Model weights
download on first use, so this cell needs internet.

In [ ]:
!pip install -q -r requirements-depth.txt

## 4 · Upload the photograph

`data/` is gitignored, so the photos are not in the clone — upload one here.
Use a shot whose mask you have already checked at home; `front.jpeg` is the
usual choice.

In [ ]:
import pathlib, shutil
from google.colab import files

pathlib.Path("data").mkdir(exist_ok=True)
print("Select your frog photo (front.jpeg or similar)...")
for name in files.upload():
    shutil.move(name, f"data/{name}")
    print("saved ->", f"data/{name}")

In [ ]:
PHOTO = "data/front.jpeg"   # <-- change if you uploaded a different filename
OUT   = "model3d/frog"
RELIEF, GRID = 0.35, 140

import pathlib
assert pathlib.Path(PHOTO).exists(), f"{PHOTO} not found — check the filename above"
print("using", PHOTO)

## 5 · Segment, then predict depth

Run as two visible steps rather than one opaque call, so the depth convention
can be checked before it becomes geometry.

**Why that check matters.** `build_mesh` writes the depth value straight into
the vertex z coordinate, and `render3d` puts the camera on the low-z side — so a
**larger value means further away**. Depth Anything predicts the opposite
(inverse depth: nearest surface scores highest). Used raw, every bump would
become a dent and the frog would render inside-out.

`orient_depth` settles it by measurement, not assumption: the frog stands in
front of its backdrop, so whichever side of the mask holds the smaller values is
the near side.

In [ ]:
import cv2, numpy as np
from reconstruct import segment_foreground, estimate_model_depth, orient_depth

bgr = cv2.imread(PHOTO)
scale = 900 / max(bgr.shape[:2])
if scale < 1:
    bgr = cv2.resize(bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

mask = segment_foreground(bgr)
print(f"foreground: {100 * mask.mean():.1f}%   (expect roughly 20-50% for a well-framed shot)")

raw = estimate_model_depth(bgr, mask=None)   # mask=None -> unoriented, as predicted
oriented, flipped = orient_depth(raw, mask)

print(f"\nraw model output   object median={np.median(raw[mask]):.3f}   "
      f"background median={np.median(raw[~mask]):.3f}")
print("flipped to the pipeline convention:", flipped)
print("(True is expected for Depth Anything, which predicts inverse depth.)")
print(f"after orienting     object median={np.median(oriented[mask]):.3f}   "
      f"background median={np.median(oriented[~mask]):.3f}   <- object must now be the SMALLER one")

### Look at the mask before trusting anything downstream

A plausible foreground percentage is **not** proof the mask is right — an
inverted mask once reported a healthy-looking 70.6% on a photo where the frog
occupied 31%, because it had latched onto the background. Look at the picture.

In the depth panel, **dark is near** and bright is far (that is the pipeline's
convention, not the usual publication one). The frog's back ridges should read
darker than the backdrop.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title("1 · photograph")
axes[1].imshow(mask, cmap="gray");                    axes[1].set_title(f"2 · mask ({100*mask.mean():.0f}% foreground)")
im = axes[2].imshow(np.where(mask, oriented, np.nan), cmap="viridis")
axes[2].set_title("3 · oriented depth (dark = near)")
fig.colorbar(im, ax=axes[2], fraction=0.046)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 6 · Mesh and export

The oriented depth is written out and passed back in via `--depth-image`, so the
network runs only once.

In [ ]:
from pathlib import Path
from reconstruct import reconstruct_photo

Path("model3d").mkdir(exist_ok=True)
depth_png = "model3d/frog_depth_input.png"
cv2.imwrite(depth_png, np.clip(oriented * 255, 0, 255).astype(np.uint8))

result = reconstruct_photo(
    Path(PHOTO), Path(OUT),
    depth_image=Path(depth_png),
    relief=RELIEF, grid=GRID,
)
print(f"mesh:  {result['vertices']} vertices, {result['faces']} triangles")
print(f"depth: {result['depth_method']}")
print(f"wrote: {result['paths']['obj']}")

## 7 · Render novel views

The real test of the reconstruction. Swing the camera and see whether the relief
holds up — the ridges should stay put and shade consistently, not swim or
invert. Keep the arc narrow: this is a relief, so a wide sweep exposes the empty
back.

In [ ]:
from render3d import render_views

view_paths, triangle_counts, elapsed = render_views(
    Path(f"{OUT}.obj"), Path("outputs/frog_views"),
    yaws=[-40, -20, 0, 20, 40], pitch=-6, size=420,
)
print(f"{len(view_paths)} views in {elapsed:.2f}s "
      f"({elapsed/len(view_paths):.2f}s per view), "
      f"triangles drawn per view: {triangle_counts}")

fig, axes = plt.subplots(1, len(view_paths), figsize=(4 * len(view_paths), 4))
for ax, path, yaw in zip(axes, view_paths, [-40, -20, 0, 20, 40]):
    ax.imshow(cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB))
    ax.set_title(f"yaw {yaw:+d}°"); ax.axis("off")
plt.tight_layout(); plt.show()

## 8 · Download

Brings back the OBJ bundle, the depth map and the rendered views. Unzip into the
project root on your laptop; `render3d.py` then works on it offline.

In [ ]:
!zip -qr frog_model.zip model3d outputs

from google.colab import files
print("model3d/ and outputs/ zipped. Unzip into the project root at home, then:")
print(f"  python render3d.py {OUT}.obj --frames 9 --sweep 80 --video --out outputs/frog")
files.download("frog_model.zip")

## 9 · Record this for the report

`CAPTURE.md` asks for the capture conditions to be written down. For this stage
note: the checkpoint (`Depth-Anything-V2-Small-hf`), that the depth is
**relative, not metric** — there is no absolute scale, so `--relief` is set by
inspection — the mesh size, and the render timing printed above.

State plainly that this is a **single-image relief**. Metric scale and a closed
surface both need the multi-view turntable capture.